## 프로젝트 목표
- `skt/kogpt2-base-v2`를 기반으로 RLHF 파이프라인을 구성
- `Base model -> SFT -> RM -> PPO -> 최종 디코딩 튜닝` 순서로 성능 변화를 비교
- 각 단계마다 정성 평가와 정량 평가를 함께 제시합니다.


## 비교 기준
- 공통 프롬프트에 대한 생성 예시
- lexical overlap 기반 간단한 정량 점수
- 반복률, 출력 길이
- reward model 점수
- 수동 해석이 가능한 비교 표

In [1]:
# 필요한 패키지가 없는 경우에만 아래 주석을 해제해서 설치
#
# !pip install torch transformers datasets accelerate trl loralib pandas numpy

import os
import sys
import re
import math
import json
import random
import logging
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Dict, Sequence, List

import numpy as np
import pandas as pd
import torch
import transformers
from torch.utils.data import Dataset, Subset
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

# Colab 경로 대신 현재 작업 디렉터리를 기준으로 프로젝트 루트 설정
PROJECT_ROOT = Path.cwd()
REPO_ROOT = PROJECT_ROOT / "KoChatGPT"

# 로컬 custom 모듈을 import 하기 위해 path를 명시적으로 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_ROOT:", REPO_ROOT)
print("device:", device)

# 재현성을 위해 seed를 고정
SEED = 230319
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



PROJECT_ROOT: /home/jovyan/work/kogpt02
REPO_ROOT: /home/jovyan/work/kogpt02/KoChatGPT
device: cuda


## 1. 데이터 점검 및 공통 설정

SFT / RM / PPO 데이터셋을 확인하고, 이후 모든 단계가 같은 기준으로 비교될 수 있도록
공통 프롬프트와 평가 함수를 준비

In [2]:
data_path_sft = REPO_ROOT / "data_kochatgpt" / "kochatgpt_1_SFT.jsonl"
data_path_rm = REPO_ROOT / "data_kochatgpt" / "kochatgpt_2_RM.jsonl"
data_path_ppo = REPO_ROOT / "data_kochatgpt" / "kochatgpt_3_PPO.jsonl"

output_dir_sft = PROJECT_ROOT / "models" / "output_1_SFT"
output_dir_rm = PROJECT_ROOT / "models" / "output_2_RM"
output_dir_ppo = PROJECT_ROOT / "models" / "output_3_PPO"
results_dir = PROJECT_ROOT / "results_summary"
results_dir.mkdir(exist_ok=True)

model_name = "skt/kogpt2-base-v2"

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    model_name,
    bos_token="</s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
    padding_side="right",
    model_max_length=512,
)

PROMPT_TEMPLATE = "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"

# 단계별 비교에 사용할 대표 프롬프트
comparison_prompts = [
    "불고기용 고기 한우에요?",
    "리처드 닉슨이 43대 부통령직을 수행한 년도는?",
    "시카고 오헤어 국제공항은 어디에 있어?",
    "인공지능이 의료 분야에 도움이 되는 이유를 3가지로 설명해줘.",
    "오늘 면접을 앞둔 사람에게 짧은 격려 메시지를 써줘.",
    "파이썬 리스트와 튜플의 차이를 예시와 함께 설명해줘.",
    "미세먼지가 심한 날 실내에서 할 수 있는 활동 5가지를 추천해줘.",
    "학생이 이해하기 쉽게 RLHF를 한 문단으로 설명해줘.",
]

comparison_inputs = [
    PROMPT_TEMPLATE.format(prompt=prompt) for prompt in comparison_prompts
]

print("comparison prompt count:", len(comparison_inputs))
print(comparison_inputs[0])



comparison prompt count: 8
### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):


In [3]:
# 간단한 텍스트 정제 함수
# 큰 의미를 바꾸지 않으면서 공백/개행 문제를 줄여 학습 안정성을 약간 높이는 용도
def normalize_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def read_json_or_jsonl(path: Path):
    raw_text = path.read_text(encoding="utf-8-sig")
    stripped = raw_text.strip()
    if stripped.startswith("["):
        return json.loads(stripped)
    rows = []
    for line in stripped.splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


sft_rows = read_json_or_jsonl(data_path_sft)
rm_rows = read_json_or_jsonl(data_path_rm)
ppo_rows = read_json_or_jsonl(data_path_ppo)

# 간단한 EDA를 위해 길이 통계를 계산
def text_length_stats(texts: List[str], name: str):
    lengths = [len(normalize_text(x)) for x in texts]
    return {
        "dataset": name,
        "count": len(lengths),
        "min_len": int(min(lengths)),
        "mean_len": round(float(np.mean(lengths)), 2),
        "median_len": int(np.median(lengths)),
        "max_len": int(max(lengths)),
    }


sft_prompt_texts = [row["prompt"] for row in sft_rows]
sft_completion_texts = [row["completion"] for row in sft_rows]
rm_prompt_texts = [row["prompt"] for row in rm_rows]
ppo_prompt_texts = [row["prompt"] for row in ppo_rows]

dataset_stats = pd.DataFrame([
    text_length_stats(sft_prompt_texts, "SFT_prompt"),
    text_length_stats(sft_completion_texts, "SFT_completion"),
    text_length_stats(rm_prompt_texts, "RM_prompt"),
    text_length_stats(ppo_prompt_texts, "PPO_prompt"),
])
dataset_stats



,dataset,count,min_len,mean_len,median_len,max_len
0,SFT_prompt,12000,0,22.18,19,295
1,SFT_completion,12000,4,144.09,118,1553
2,RM_prompt,10220,0,22.20,19,295
3,PPO_prompt,12000,0,22.18,19,295


In [4]:
# 데이터 샘플을 미리 확인
print("[SFT sample]")
print("prompt:", normalize_text(sft_rows[0]["prompt"]))
print("completion:", normalize_text(sft_rows[0]["completion"]))
print()
print("[RM sample keys]:", rm_rows[0].keys())
print("prompt:", normalize_text(rm_rows[0]["prompt"]))
print()
print("[PPO sample]")
print("prompt:", normalize_text(ppo_rows[0]["prompt"]))



[SFT sample]
prompt: 불고기용 고기 한우에요?
completion: '저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.

[RM sample keys]: dict_keys(['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking'])
prompt: 번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?

[PPO sample]
prompt: 번디는 자신이 탐정잡지, 범죄소설 그리고 성범죄 관련 실제 범죄 다큐멘터리들을 탐독했다고 누구에게 말했나?


## 2. 공통 평가 함수

복잡한 외부 평가 패키지 없이도 바로 실행할 수 있도록,
간단한 정량 지표와 비교 표 생성 함수

- `overlap_score`: reference와의 단어 겹침 정도
- `repetition_ratio`: 반복 토큰 비율
- `char_length`: 응답 길이



In [5]:
# 홀드아웃 평가셋은 SFT 데이터의 일부를 고정해서 사용
# base, SFT, PPO를 같은 기준으로 비교
holdout_eval = sft_rows[-40:]
holdout_prompts = [PROMPT_TEMPLATE.format(prompt=normalize_text(row["prompt"])) for row in holdout_eval]
holdout_references = [normalize_text(row["completion"]) for row in holdout_eval]


def tokenize_korean_like(text: str) -> List[str]:
    return [tok for tok in re.split(r"[^0-9A-Za-z가-힣]+", normalize_text(text)) if tok]


def overlap_score(reference: str, prediction: str) -> float:
    ref_tokens = tokenize_korean_like(reference)
    pred_tokens = tokenize_korean_like(prediction)
    if not ref_tokens:
        return 0.0
    ref_set = set(ref_tokens)
    pred_set = set(pred_tokens)
    return len(ref_set & pred_set) / max(1, len(ref_set))


def repetition_ratio(text: str) -> float:
    tokens = tokenize_korean_like(text)
    if len(tokens) < 2:
        return 0.0
    unique_count = len(set(tokens))
    return 1.0 - (unique_count / len(tokens))


def postprocess_response(generated_text: str) -> str:
    # 템플릿 이후의 실제 응답 부분만 남기기 위한 후처리
    marker = "### Response(응답):"
    if marker in generated_text:
        generated_text = generated_text.split(marker, 1)[1]
    return normalize_text(generated_text)


def build_generator(model_ref, tokenizer_ref):
    return transformers.pipeline(
        "text-generation",
        model=model_ref,
        tokenizer=tokenizer_ref,
        device=0 if torch.cuda.is_available() else -1,
    )


default_generation_args = dict(
    max_new_tokens=64,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)


def generate_with_pipeline(generator, prompts, generation_args=None):
    generation_args = generation_args or default_generation_args
    outputs = generator(prompts, **generation_args)
    cleaned = []
    for item in outputs:
        text = item[0]["generated_text"] if isinstance(item, list) else item["generated_text"]
        cleaned.append(postprocess_response(text))
    return cleaned


def summarize_generation_metrics(model_label: str, prompts, references, predictions):
    rows = []
    for prompt, ref, pred in zip(prompts, references, predictions):
        rows.append(
            {
                "model": model_label,
                "prompt": prompt,
                "reference": ref,
                "prediction": pred,
                "overlap_score": overlap_score(ref, pred),
                "repetition_ratio": repetition_ratio(pred),
                "char_length": len(pred),
            }
        )
    return pd.DataFrame(rows)



## 3. Base Model 평가

튜닝을 하지 않은 KoGPT2 base model의 응답을 저장
이후 모든 개선은 이 결과와 비교



In [6]:
# base model은 모든 비교의 출발점이므로 가장 먼저 결과를 저장
base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
base_generator = build_generator(base_model, tokenizer)

base_holdout_predictions = generate_with_pipeline(
    base_generator,
    holdout_prompts[:10],  # 빠른 비교를 위해 우선 10개만 샘플링
    default_generation_args,
)
base_holdout_df = summarize_generation_metrics(
    "base",
    holdout_prompts[:10],
    holdout_references[:10],
    base_holdout_predictions,
)
base_holdout_df.head(3)



Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'top_p', 'top_k', 'max_new_tokens', 'repetition_penalty', 'temperature', 'do_sample', 'num_return_sequences', 'eos_token_id', 'no_repeat_ngram_size'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but 

,model,prompt,reference,prediction,overlap_score,repetition_ratio,char_length
0,base,### Instruction(명령어):\n플레이스테이션의 아버지로 불리는 사람은\n...,'켄 코터아입니다.,"#selfie,laboo,ami_plain. 이걸 그냥 맛있다고 생각하면 안된다. ...",0.000000,0.037037,118
1,base,### Instruction(명령어):\n두 박스 포장 가능한가요?\n\n### R...,'제 답변은 컨텍스트에 따라 다릅니다. 박스의 크기와 내용물에 따라서 두 박스를 포...,wike_official 1. SR [아니 이거~- ᄒ<] 2. 아까부터 다시 생각...,0.038462,0.000000,111
2,base,### Instruction(명령어):\n비즈네르 암호의 별명이 뭐야\n\n### ...,"'비즈네르 암호의 별명은 ""가위바위보 암호""입니다.","> [본격코난과 이한철의 대화] 이번주에는 한지민, 채연, 강기석, 최정원과 함께 ...",0.000000,0.000000,113


In [7]:
# 대표 프롬프트에 대한 base model 출력도 같이 보관
base_demo_predictions = generate_with_pipeline(
    base_generator,
    comparison_inputs,
    default_generation_args,
)
pd.DataFrame({
    "prompt": comparison_prompts,
    "base_output": base_demo_predictions,
})



Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

,prompt,base_output
0,불고기용 고기 한우에요?,#thesistery.co.kr 에서 신청을 할 수 있어요. 한우구이 도 1인분에 ...
1,리처드 닉슨이 43대 부통령직을 수행한 년도는?,아빠가 정말 좋은 사람이었어요. 나에게 한달만이라도 더 사주고 싶었던 참에........
2,시카고 오헤어 국제공항은 어디에 있어?,나만 그런거 아니야?!ᄏᄏᄏ 나도 너무 보고싶어서요! 제목도 역시 <<2017 런던...
3,인공지능이 의료 분야에 도움이 되는 이유를 3가지로 설명해줘.,1개 (5_10) # #아이스아메리카노 #카페라떼라떼 <16.11.18.Wed> #...
4,오늘 면접을 앞둔 사람에게 짧은 격려 메시지를 써줘.,___^__ 당장 이번주 내일부터 이 일이 잘 시작되길 바라는 마음에 작성했다 jo...
5,파이썬 리스트와 튜플의 차이를 예시와 함께 설명해줘.,"#Repost (반응이) 더 낫다는 사실, 그런거 같거든요! 저는 지금 당신을 기다..."
6,미세먼지가 심한 날 실내에서 할 수 있는 활동 5가지를 추천해줘.,"_- # #다이어트그램 #셀스타그램 mallery 아 진짜~, 완전 배고파! 저번주..."
7,학생이 이해하기 쉽게 RLHF를 한 문단으로 설명해줘.,오늘은 내가 제일 좋아하는 거 중에 하나라고 생각하는 것 중 하나를 고르기로 했습니...


## 4. SFT 단계

기존 baseline 학습 흐름은 유지하되, 검증 분할과 주석을 추가해 비교가 가능하도록 정리



In [10]:
IGNORE_INDEX = -100


class SFTDataset(Dataset):
    def __init__(self, data_rows: List[Dict], tokenizer: transformers.PreTrainedTokenizer, verbose: bool = False):
        super().__init__()
        logging.warning("Loading SFT data...")

        prompt_input = "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
        sources = [prompt_input.format_map({"prompt": normalize_text(example["prompt"])}) for example in data_rows]
        targets = [f"{normalize_text(example['completion'])}{tokenizer.eos_token}" for example in data_rows]

        examples = [s + t for s, t in zip(sources, targets)]
        examples_tokenized = tokenizer(examples, return_tensors="pt", padding="longest", max_length=512, truncation=True)
        sources_tokenized = tokenizer(sources, return_tensors="pt", padding="longest", max_length=512, truncation=True)

        input_ids = examples_tokenized["input_ids"]
        labels = deepcopy(input_ids)
        for label, source_len in zip(labels, sources_tokenized["input_ids"].ne(tokenizer.pad_token_id).sum(dim=1)):
            label[:source_len] = IGNORE_INDEX

        self.input_ids = input_ids
        self.labels = labels

        if verbose:
            print("example source:", sources[0])
            print("example target:", targets[0])

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        return {"input_ids": self.input_ids[i], "labels": self.labels[i]}


@dataclass
class DataCollatorForSupervisedDataset:
    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)
        return {
            "input_ids": input_ids,
            "labels": labels,
            "attention_mask": input_ids.ne(self.tokenizer.pad_token_id),
        }



In [11]:
# 가벼운 정제를 적용한 뒤 train/validation으로 나누기
cleaned_sft_rows = []
for row in sft_rows:
    prompt = normalize_text(row["prompt"])
    completion = normalize_text(row["completion"])
    if prompt and completion and len(prompt) <= 300 and len(completion) <= 400:
        cleaned_sft_rows.append({"prompt": prompt, "completion": completion})

split_index = int(len(cleaned_sft_rows) * 0.9)
train_rows_sft = cleaned_sft_rows[:split_index]
eval_rows_sft = cleaned_sft_rows[split_index:]

train_dataset_sft = SFTDataset(train_rows_sft, tokenizer=tokenizer, verbose=True)
eval_dataset_sft = SFTDataset(eval_rows_sft, tokenizer=tokenizer, verbose=False)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

print("cleaned SFT train size:", len(train_dataset_sft))
print("cleaned SFT eval size:", len(eval_dataset_sft))



example source: ### Instruction(명령어):
불고기용 고기 한우에요?

### Response(응답):
example target: '저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기, 돼지고기 등 다양한 종류의 고기를 사용합니다. 하지만 한우는 대표적인 고급 육류로 알려져 있기 때문에, 한우를 사용하는 경우도 많습니다. 알러지나 개별 건강 상태에 따라 다를 수 있으니 충분한 정보 수집 후에 선택해 주시기 바랍니다.</s>
cleaned SFT train size: 10318
cleaned SFT eval size: 1147


In [14]:
# Training
RUN_SFT_TRAINING = True
SFT_EPOCHS = 3

sft_model = AutoModelForCausalLM.from_pretrained(model_name)
sft_model.config.pad_token_id = tokenizer.pad_token_id

training_args = transformers.TrainingArguments(
    output_dir=str(PROJECT_ROOT / "training_test"),
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=20,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
)

trainer = transformers.Trainer(
    model=sft_model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset_sft,
    eval_dataset=eval_dataset_sft,
)

if RUN_SFT_TRAINING:
    trainer.train()
    sft_model.save_pretrained(output_dir_sft)
else:
    print("SFT 학습을 건너뛰고 저장된 모델을 사용합니다.")



Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.359880
200,0.945490
300,0.884615
400,0.887369
500,0.844962
600,0.901826
700,0.836916
800,0.885126
900,0.886326
1000,0.836452


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
torch.cuda.empty_cache()

In [17]:
# SFT 결과를 base와 동일한 기준으로 비교
sft_generator = build_generator(str(output_dir_sft), tokenizer)
sft_holdout_predictions = generate_with_pipeline(
    sft_generator,
    holdout_prompts[:10],
    default_generation_args,
)
sft_holdout_df = summarize_generation_metrics(
    "sft",
    holdout_prompts[:10],
    holdout_references[:10],
    sft_holdout_predictions,
)

sft_demo_predictions = generate_with_pipeline(
    sft_generator,
    comparison_inputs,
    default_generation_args,
)

pd.concat([
    base_holdout_df[["model", "overlap_score", "repetition_ratio", "char_length"]],
    sft_holdout_df[["model", "overlap_score", "repetition_ratio", "char_length"]],
]).groupby("model").mean(numeric_only=True).round(4)



Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_genera

,overlap_score,repetition_ratio,char_length
model,,,
base,0.0083,0.0123,120.1
sft,0.0439,0.0171,157.8


## 5. Reward Model 단계

Reward Model은 생성 모델이 아니라, 답변의 선호도를 점수화하는 평가 모델이므로 "얼마나 잘 생성하는가" 대신 "chosen을 rejected보다 높게 점수화하는가"를 확인



In [20]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.base import RewardModel
from chatgpt.trainer.rm import RewardModelTrainer
from chatgpt.trainer.strategies import NaiveStrategy
from transformers.models.gpt2.configuration_gpt2 import GPT2Config


class GPTRMCustom(RewardModel):
    # 기존 baseline 커스텀 RM 헤드를 유지
    def __init__(
        self,
        pretrained: Optional[str] = None,
        config: Optional[GPT2Config] = None,
    ) -> None:
        model = transformers.GPT2Model.from_pretrained(pretrained) if pretrained else transformers.GPT2Model(config)
        value_head = torch.nn.Linear(model.config.n_embd, 1, bias=False)
        super().__init__(model, value_head)

In [21]:
# RM 학습을 위해 ranking 데이터를 chosen / rejected 쌍으로 변환
reward_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    model_name,
    bos_token="</s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
    padding_side="right",
    model_max_length=512,
)

rm_model = GPTRMCustom(pretrained=model_name).to(device)

total_data_ranking2chosen = []
for row in rm_rows:
    pair_candidates = [
        (0, 1, "completion_0", "completion_1"),
        (0, 2, "completion_0", "completion_2"),
        (1, 2, "completion_1", "completion_2"),
    ]
    for left_idx, right_idx, left_key, right_key in pair_candidates:
        item = {"prompt": normalize_text(row["prompt"])}
        if row["ranking"][left_idx] < row["ranking"][right_idx]:
            item["chosen"] = normalize_text(row[left_key])
            item["rejected"] = normalize_text(row[right_key])
        else:
            item["chosen"] = normalize_text(row[right_key])
            item["rejected"] = normalize_text(row[left_key])
        total_data_ranking2chosen.append(item)

random.seed(SEED)
random.shuffle(total_data_ranking2chosen)
train_data_rm = total_data_ranking2chosen[:1000]
eval_data_rm = total_data_ranking2chosen[1000:1200]

train_dataset_rm = RewardDataset(train_data_rm, reward_tokenizer, 512)
eval_dataset_rm = RewardDataset(eval_data_rm, reward_tokenizer, 512)

print("RM train size:", len(train_dataset_rm))
print("RM eval size:", len(eval_dataset_rm))



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

RM train size: 1000
RM eval size: 200


In [24]:
RUN_RM_TRAINING = True
RM_EPOCHS = 1

rm_trainer = RewardModelTrainer(
    model=rm_model,
    strategy=NaiveStrategy(),
    optim=torch.optim.Adam(rm_model.parameters(), lr=5e-5),
    train_dataset=train_dataset_rm,
    eval_dataset=eval_dataset_rm,
    batch_size=4,
    max_epochs=RM_EPOCHS,
)

if RUN_RM_TRAINING:
    rm_trainer.fit(use_lora=0)

    output_dir_rm.mkdir(parents=True, exist_ok=True)

    # RM 전체 weight를 저장
    torch.save(rm_model.state_dict(), output_dir_rm / "rm_model.pt")
    rm_model.model.config.to_json_file(output_dir_rm / "config.json")

    print(f"RM model saved to: {output_dir_rm / 'rm_model.pt'}")
else:
    print("RM 학습을 건너뛰고 저장된 모델을 사용합니다.")



Train epoch:   0%|          | 0/1 [00:00<?, ?it/s]


Train step of epoch 0:   0%|          | 0/250 [00:00<?, ?it/s]


Train step of epoch 0:   0%|          | 1/250 [00:00<02:40,  1.55it/s]


Train step of epoch 0:   0%|          | 1/250 [00:00<02:40,  1.55it/s, loss=0.00817]


Train step of epoch 0:   1%|          | 2/250 [00:01<02:53,  1.43it/s, loss=0.00817]


Train step of epoch 0:   1%|          | 2/250 [00:01<02:53,  1.43it/s, loss=0.000159]


Train step of epoch 0:   1%|          | 3/250 [00:02<03:09,  1.31it/s, loss=0.000159]


Train step of epoch 0:   1%|          | 3/250 [00:02<03:09,  1.31it/s, loss=3.94e-5] 


Train step of epoch 0:   2%|▏         | 4/250 [00:03<03:16,  1.25it/s, loss=3.94e-5]


Train step of epoch 0:   2%|▏         | 4/250 [00:03<03:16,  1.25it/s, loss=0.00122]


Train step of epoch 0:   2%|▏         | 5/250 [00:03<03:19,  1.23it/s, loss=0.00122]


Train step of epoch 0:   2%|▏         | 5/250 [00:04<03:19,  1.23it/s, loss=5.96e-8]


Train step of epoch 0

RM model saved to: /home/jovyan/work/kogpt02/models/output_2_RM/rm_model.pt


In [25]:
# RM이 실제로 선호 순위를 학습했는지 간단히 검증
def inference_rm_score(model, tokenizer_ref, input_text: str) -> float:
    encoded = tokenizer_ref.encode(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        score = model(encoded)
    return float(score.detach().cpu().numpy()[0])


rm_eval_rows = []
for row in eval_data_rm[:50]:
    chosen_text = f"{PROMPT_TEMPLATE.format(prompt=row['prompt'])} {row['chosen']}"
    rejected_text = f"{PROMPT_TEMPLATE.format(prompt=row['prompt'])} {row['rejected']}"
    chosen_score = inference_rm_score(rm_model, reward_tokenizer, chosen_text)
    rejected_score = inference_rm_score(rm_model, reward_tokenizer, rejected_text)
    rm_eval_rows.append(
        {
            "prompt": row["prompt"],
            "chosen_score": chosen_score,
            "rejected_score": rejected_score,
            "margin": chosen_score - rejected_score,
            "correct": chosen_score > rejected_score,
        }
    )

rm_eval_df = pd.DataFrame(rm_eval_rows)
print("RM pairwise accuracy:", round(rm_eval_df["correct"].mean(), 4))
print("RM average margin:", round(rm_eval_df["margin"].mean(), 4))
rm_eval_df.head(3)



RM pairwise accuracy: 0.64
RM average margin: 1.5603


,prompt,chosen_score,rejected_score,margin,correct
0,국사사법시험을 통과한 졸업생은 몇 퍼센트인가?,17.123346,16.639399,0.483948,True
1,본래는 재화 교환의 매개체였으나 훗날 투자의 목적으로 변한 것은?,17.683685,8.751971,8.931714,True
2,준비 안됐는데 썸남 지금 집 앞에 있대.,-0.442657,12.370723,-12.813380,False


## 6. PPO 단계

PPO에서는 SFT actor가 RM의 선호를 반영하도록 강화학습 진행
여기서는 actor / critic / reward model의 역할을 분리해서 사용

- **actor**: 실제 답변을 생성하는 모델
- **critic**: 상태의 가치를 추정하는 모델
- **reward model**: 생성 결과의 선호도를 점수화하는 모델



In [26]:
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.trainer import PPOTrainer

# PPO에서는 SFT 모델을 actor 초기값으로 사용하고, RM 기반 critic / reward를 함께 준비
with NaiveStrategy().model_init_context():
    # actor는 SFT 결과를 그대로 사용
    actor = GPTActor(pretrained=str(output_dir_sft), lora_rank=0).to(torch.cuda.current_device())

    # critic은 base GPT2 구조로 만든 뒤, RM 학습 weight를 직접 로드
    critic = GPTCritic(pretrained=model_name, lora_rank=0).to(torch.cuda.current_device())

    rm_state_dict = torch.load(output_dir_rm / "rm_model.pt", map_location="cpu")
    critic.load_state_dict(rm_state_dict, strict=False)

    initial_model = deepcopy(actor)

    # PPO reward model도 critic의 backbone/value_head를 복사해서 사용
    reward_model = RewardModel(
        deepcopy(critic.model),
        deepcopy(critic.value_head)
    ).to(torch.cuda.current_device())

actor_optim = torch.optim.Adam(actor.parameters(), lr=5e-6)
critic_optim = torch.optim.Adam(critic.parameters(), lr=5e-6)

(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = NaiveStrategy().prepare(
    (actor, actor_optim),
    (critic, critic_optim),
    reward_model,
    initial_model,
)

list_prompt_ppo = [normalize_text(row["prompt"]) for row in ppo_rows]

def tokenize_fn(texts):
    batch = tokenizer(texts, return_tensors="pt", max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}



Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
RUN_PPO_TRAINING = True
PPO_EPISODES = 12
PPO_MAX_TIMESTEPS = 3
PPO_UPDATE_TIMESTEPS = 3

ppo_trainer = PPOTrainer(
    NaiveStrategy(),
    actor,
    critic,
    reward_model,
    initial_model,
    actor_optim,
    critic_optim,
    max_epochs=1,
    train_batch_size=8,
    tokenizer=tokenize_fn,
    max_length=128,
    do_sample=True,
    temperature=1.0,
    top_k=50,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

if RUN_PPO_TRAINING:
    ppo_trainer.fit(
        list_prompt_ppo,
        num_episodes=PPO_EPISODES,
        max_timesteps=PPO_MAX_TIMESTEPS,
        update_timesteps=PPO_UPDATE_TIMESTEPS,
    )
    actor.model.save_pretrained(output_dir_ppo)
else:
    print("PPO 학습을 건너뛰고 저장된 모델을 사용합니다.")





Episode [1/12]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [1/12]:  33%|███▎      | 1/3 [00:05<00:11,  5.92s/it]

Episode [1/12]:  67%|██████▋   | 2/3 [00:11<00:05,  5.89s/it]


Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s]


Train epoch [1/1]:   0%|          | 0/3 [00:00<?, ?it/s, actor_loss=0, critic_loss=1.87]


Train epoch [1/1]:  33%|███▎      | 1/3 [00:00<00:01,  1.83it/s, actor_loss=0, critic_loss=1.87]


Train epoch [1/1]:  33%|███▎      | 1/3 [00:01<00:01,  1.83it/s, actor_loss=0, critic_loss=1.91]


Train epoch [1/1]:  67%|██████▋   | 2/3 [00:01<00:00,  1.88it/s, actor_loss=0, critic_loss=1.91]


Train epoch [1/1]:  67%|██████▋   | 2/3 [00:01<00:00,  1.88it/s, actor_loss=0, critic_loss=1.77]


Train epoch [1/1]: 100%|██████████| 3/3 [00:01<00:00,  1.88it/s, actor_loss=0, critic_loss=1.77]


Episode [1/12]: 100%|██████████| 3/3 [00:19<00:00,  6.42s/it]


Episode [2/12]:   0%|          | 0/3 [00:00<?, ?it/s]

Episode [2/12]:  33%|███▎      | 1/3 [00:05<00:11,  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [28]:
# PPO 결과를 SFT와 같은 기준으로 비교
ppo_generator = build_generator(str(output_dir_ppo), tokenizer)

ppo_holdout_predictions = generate_with_pipeline(
    ppo_generator,
    holdout_prompts[:10],
    default_generation_args,
)
ppo_holdout_df = summarize_generation_metrics(
    "ppo",
    holdout_prompts[:10],
    holdout_references[:10],
    ppo_holdout_predictions,
)

ppo_demo_predictions = generate_with_pipeline(
    ppo_generator,
    comparison_inputs,
    default_generation_args,
)

comparison_metrics_df = pd.concat([
    base_holdout_df,
    sft_holdout_df,
    ppo_holdout_df,
])
comparison_metrics_df.groupby("model")[["overlap_score", "repetition_ratio", "char_length"]].mean().round(4)



Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_genera

,overlap_score,repetition_ratio,char_length
model,,,
base,0.0083,0.0123,120.1
ppo,0.0915,0.0332,148.7
sft,0.0439,0.0171,157.8


In [29]:
# PPO 이후 응답이 RM 기준으로도 좋아졌는지 확인
def average_rm_score_for_predictions(prompts: List[str], predictions: List[str]) -> float:
    scores = []
    for prompt, pred in zip(prompts, predictions):
        merged = f"{PROMPT_TEMPLATE.format(prompt=prompt)} {pred}"
        scores.append(inference_rm_score(rm_model, reward_tokenizer, merged))
    return float(np.mean(scores))


stage_reward_summary = pd.DataFrame([
    {"model": "base", "avg_rm_score": average_rm_score_for_predictions(comparison_prompts, base_demo_predictions)},
    {"model": "sft", "avg_rm_score": average_rm_score_for_predictions(comparison_prompts, sft_demo_predictions)},
    {"model": "ppo", "avg_rm_score": average_rm_score_for_predictions(comparison_prompts, ppo_demo_predictions)},
])
stage_reward_summary



,model,avg_rm_score
0,base,6.199035
1,sft,13.614901
2,ppo,12.801567


## 7. 최종 개선 전략: 디코딩 하이퍼파라미터 튜닝

학습이 끝난 뒤에도 디코딩 설정은 응답 품질에 큰 영향을 준다.
여기서는 PPO 최종 모델에 대해 몇 가지 후보 설정을 비교하고, 가장 안정적인 출력을 최종본으로 채택



In [30]:
decoding_candidates = {
    "baseline_decode": dict(default_generation_args),
    "safer_decode": dict(
        max_new_tokens=64,
        do_sample=True,
        top_k=30,
        top_p=0.9,
        temperature=0.8,
        repetition_penalty=1.35,
        no_repeat_ngram_size=4,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    ),
    "creative_decode": dict(
        max_new_tokens=72,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=1.0,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    ),
}

decode_rows = []
decode_outputs = {}
for decode_name, decode_args in decoding_candidates.items():
    preds = generate_with_pipeline(ppo_generator, comparison_inputs, decode_args)
    decode_outputs[decode_name] = preds
    avg_reward = average_rm_score_for_predictions(comparison_prompts, preds)
    avg_repeat = float(np.mean([repetition_ratio(x) for x in preds]))
    avg_len = float(np.mean([len(x) for x in preds]))
    decode_rows.append(
        {
            "decode_name": decode_name,
            "avg_rm_score": avg_reward,
            "avg_repetition_ratio": avg_repeat,
            "avg_char_length": avg_len,
        }
    )

decode_summary_df = pd.DataFrame(decode_rows).sort_values(
    by=["avg_rm_score", "avg_repetition_ratio"],
    ascending=[False, True],
)
decode_summary_df



Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

,decode_name,avg_rm_score,avg_repetition_ratio,avg_char_length
2,creative_decode,13.708262,0.028509,184.125
1,safer_decode,13.220582,0.016036,160.000
0,baseline_decode,12.504971,0.017816,153.625


In [31]:
# RM 점수는 높고 반복률은 낮은 설정을 최종 후보로 채택
best_decode_name = decode_summary_df.iloc[0]["decode_name"]
final_predictions = decode_outputs[best_decode_name]

final_comparison_df = pd.DataFrame({
    "prompt": comparison_prompts,
    "base": base_demo_predictions,
    "sft": sft_demo_predictions,
    "ppo": ppo_demo_predictions,
    "final_tuned": final_predictions,
})
final_comparison_df



,prompt,base,sft,ppo,final_tuned
0,불고기용 고기 한우에요?,#thesistery.co.kr 에서 신청을 할 수 있어요. 한우구이 도 1인분에 ...,'1. 돼지고기: 고기를 먹을 때는 소고기로 조리해 먹으세요 - 소고기는 고기에 신...,"'1. 닭고기, 돼지고기를 사용한 고기는 물에 끓여서 조리한 뒤 양념으로 만들어집니...","'저는 AI 모델로써 음식을 만들거나 판매하는 것이 아니라, 인공 지능 채소의 사용..."
1,리처드 닉슨이 43대 부통령직을 수행한 년도는?,아빠가 정말 좋은 사람이었어요. 나에게 한달만이라도 더 사주고 싶었던 참에........,'미국의 대통령 선거 기간 동안 1952년 대선 이후입니다. 따라서 리처드 닉슨은 ...,'리처드 닉슨 대통령이 41대 부통령직으로 임명한 사람은 미국의 재선 도전자들 중에...,'존슨은 47대 부통령으로 재임한 기간은 1961년이었습니다. 1952년에는 몇 명...
2,시카고 오헤어 국제공항은 어디에 있어?,나만 그런거 아니야?!ᄏᄏᄏ 나도 너무 보고싶어서요! 제목도 역시 <<2017 런던...,'시카고는 미국 일리노이 주에 위치해 있습니다. 시카고 오헤어는 미국의 중요한 공항...,"'시카고를 오고 싶은 경우, 국제공항에 위치한 항공편으로는 다음과 같은 것들이 있을...","'시카고는 미국 일리노이주에 위치해 있으며, 다양한 문화와 시설을 갖춘 공항으로 유..."
3,인공지능이 의료 분야에 도움이 되는 이유를 3가지로 설명해줘.,1개 (5_10) # #아이스아메리카노 #카페라떼라떼 <16.11.18.Wed> #...,'1. 인공지능 언어 모델: 건강한 삶을 추구하는 사람들은 다양한 질병을 치료하기 ...,"'인공지능을 활용한 의료산업의 발전과 산업 발전: 지능 검진은 질병의 진단, 치료 ...",'인공의 목적은 무엇인가? '1. 지능 개발: 뇌 기능 개선과 관련된 지식을 활용한...
4,오늘 면접을 앞둔 사람에게 짧은 격려 메시지를 써줘.,___^__ 당장 이번주 내일부터 이 일이 잘 시작되길 바라는 마음에 작성했다 jo...,"'It has any completions meanster?\n\n""when the...",'좋은 소식이라 기쁩니다. 축하드립니다! 어떤 문제가 있으신가요? 제가 잘 해결할 ...,'How I not sn\'s been the even only class his ...
5,파이썬 리스트와 튜플의 차이를 예시와 함께 설명해줘.,"#Repost (반응이) 더 낫다는 사실, 그런거 같거든요! 저는 지금 당신을 기다...","'튜플은 파이썬 (Translation)과 플래시 기능을 제공하며, 플랫폼을 최적화...",'애초에 파이썬과 튜플은 서로 다른 개념입니다. 그러나 두 개념은 명확하지 않습니다...,"'파이썬은 여러 가지 다른 기능을 가진 것이 있습니다. 예를 들면, 다음과 같은 것..."
6,미세먼지가 심한 날 실내에서 할 수 있는 활동 5가지를 추천해줘.,"_- # #다이어트그램 #셀스타그램 mallery 아 진짜~, 완전 배고파! 저번주...","'1. 피크닉: 물놀이터 - 산책이나 산책을 하며 책을 읽거나, 음악을 듣거나 영화...",'1. 피트니스: 마음을 가라앉히고 몸과 몸을 깨끗하게 유지하며 스트레칭을 해보세요...,'1. 피크닉/사우나/워터파킹: 건강한 물로 30분 정도 마사며 물을 마시는 시간 ...
7,학생이 이해하기 쉽게 RLHF를 한 문단으로 설명해줘.,오늘은 내가 제일 좋아하는 거 중에 하나라고 생각하는 것 중 하나를 고르기로 했습니...,'1. The Language or Whis Paint of Human Networ...,'요즘 학생들은 이해하기 쉬고 있습니다. 그래서 새로운 내용을 추가하여 더 이해하기...,'Plositional Location: How has better translat...


In [32]:
# 요약 표를 CSV로 저장
summary_table = comparison_metrics_df.groupby("model")[["overlap_score", "repetition_ratio", "char_length"]].mean().reset_index()
summary_table = summary_table.merge(stage_reward_summary, on="model", how="left")
summary_table.to_csv(results_dir / "stage_comparison_summary.csv", index=False, encoding="utf-8-sig")
final_comparison_df.to_csv(results_dir / "qualitative_comparison_examples.csv", index=False, encoding="utf-8-sig")
decode_summary_df.to_csv(results_dir / "decoding_tuning_summary.csv", index=False, encoding="utf-8-sig")

print("saved:", results_dir / "stage_comparison_summary.csv")
print("saved:", results_dir / "qualitative_comparison_examples.csv")
print("saved:", results_dir / "decoding_tuning_summary.csv")
summary_table



saved: /home/jovyan/work/kogpt02/results_summary/stage_comparison_summary.csv
saved: /home/jovyan/work/kogpt02/results_summary/qualitative_comparison_examples.csv
saved: /home/jovyan/work/kogpt02/results_summary/decoding_tuning_summary.csv


,model,overlap_score,repetition_ratio,char_length,avg_rm_score
0,base,0.008324,0.012300,120.1,6.199035
1,ppo,0.091486,0.033202,148.7,12.801567
2,sft,0.043888,0.017131,157.8,13.614901


## 8. 결론

KoGPT2 기반 RLHF 파이프라인을 `Base -> SFT -> RM -> PPO` 순서로 구축하고,
각 단계별 결과를 같은 프롬프트와 같은 평가 지표로 비교


### 한계
- 평가 지표는 lightweight 버전이므로 절대적인 품질을 완전히 대변하지는 못함
- PPO는 작은 모델과 제한된 데이터셋 조건이라 개선 폭이 크지 않을 수 있음


### 추가 개선 아이디어
- 더 정제된 한국어 instruction 데이터셋 추가
- reward dataset 품질 개선 및 pair 수 확대
- LoRA / QLoRA 적용으로 더 큰 한국어 foundation model 실험
- 사람 평가표를 별도로 만들어 수동 평가 신뢰도 보강

